# Modeling 7.1: California training, all red/white validation

Train **all four model types** on 20,000 labeled **California** red/white reviews: 13,000 red and 7,000 white. The 10,000 training reviews from Model 6 are retained, and 10,000 new California reviews are added. Validate on **every red/white review from every region** in saved `validation.parquet`. This tests whether models trained only on California transfer to other regions. Training is no longer balanced because California has fewer than 10,000 white training reviews; the 13,000/7,000 mix is held identical in Model 7.2.

The three TF-IDF models run locally. DistilBERT trains on a Google Colab GPU and exports a compact results ZIP that this notebook reads back into the same comparison table. No dummy baseline is included. Only `description` is a model feature; metadata determines the sample and labels. `test.parquet` is never loaded.

**Workflow:** Run Sections 1–3 locally. Open this same notebook in Colab, select a T4 GPU, upload `train.parquet` and `validation.parquet`, and run Sections 1–4. Download `modeling_7_1_distilbert_results.zip`, put it beside this notebook locally, and run Section 5 to verify the data and add DistilBERT to the table. If Colab installs or upgrades packages, restart its runtime and rerun from the top.

## 1. Load the saved partitions

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
LABELS = ['red', 'white']
if not all(Path(name).exists() for name in ['train.parquet', 'validation.parquet']):
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Place train.parquet and validation.parquet beside this notebook.') from exc
    print('Upload train.parquet and validation.parquet together.')
    files.upload()
assert all(Path(name).exists() for name in ['train.parquet', 'validation.parquet'])
saved_train = pd.read_parquet('train.parquet')
saved_validation = pd.read_parquet('validation.parquet')
assert saved_train.index.intersection(saved_validation.index).empty
print(f'Saved partitions: {len(saved_train):,} train; {len(saved_validation):,} validation')

Saved partitions: 101,506 train; 33,835 validation


## 2. Select California training reviews and the full red/white validation set

Normalize review descriptions, exclude conflicts and duplicates, and remove training copies of text also found in validation. Reconstruct Model 6's 5,000 red and 5,000 white training reviews, then add 8,000 red and 2,000 white California reviews. The validation set includes California **and non-California** red/white reviews. It keeps the natural class mix.

In [2]:
N_TRAIN = 20_000
MODEL_NUMBER = '7_1'
TRAIN_COUNTS = {'red': 13_000, 'white': 7_000}

def redwhite_pool(frame, california_only=False):
    mask = frame['color'].isin(LABELS)
    if california_only:
        mask = mask & frame['province'].eq('California')
    pool = frame.loc[mask].dropna(subset=['description', 'color']).copy()
    pool['description_key'] = pool['description'].str.replace(r'\s+', ' ', regex=True).str.strip().str.casefold()
    pool = pool.loc[pool['description_key'].ne('')]
    conflicts = pool.groupby('description_key')['color'].nunique()
    pool = pool.loc[~pool['description_key'].isin(conflicts[conflicts > 1].index)]
    return pool.drop_duplicates('description_key')

train_pool = redwhite_pool(saved_train, california_only=True)
validation_all = redwhite_pool(saved_validation)
validation_keys = set(validation_all['description_key'])
cross_split_keys = set(train_pool['description_key']) & validation_keys
train_pool = train_pool.loc[~train_pool['description_key'].isin(cross_split_keys)].copy()
assert set(train_pool['description_key']).isdisjoint(validation_keys)
assert train_pool.index.intersection(validation_all.index).empty
assert all(train_pool['color'].eq(color).sum() >= target for color, target in TRAIN_COUNTS.items())

training_parts = []
model6_parts = []
for color in LABELS:
    color_pool = train_pool.loc[train_pool['color'].eq(color)]
    original_500 = color_pool.sample(n=500, random_state=SEED)
    remaining = color_pool.loc[~color_pool.index.isin(original_500.index)]
    next_4500 = remaining.sample(n=4_500, random_state=SEED)
    model6_parts.extend([original_500, next_4500])
    remaining = remaining.loc[~remaining.index.isin(next_4500.index)]
    additional = remaining.sample(n=TRAIN_COUNTS[color] - 5_000, random_state=SEED)
    training_parts.extend([original_500, next_4500, additional])
model6_indices = set(pd.concat(model6_parts).index)
train_sample = pd.concat(training_parts).sample(frac=1, random_state=SEED)
assert model6_indices.issubset(set(train_sample.index))
print('Model 6 training rows retained:', len(model6_indices))
assert len(train_sample) == N_TRAIN and train_sample.index.is_unique
assert train_sample['color'].value_counts().to_dict() == TRAIN_COUNTS
assert set(train_sample['description_key']).isdisjoint(validation_keys)
print('Cross-split normalized descriptions excluded from training:', len(cross_split_keys))
display(pd.DataFrame({'training': train_sample['color'].value_counts(), 'validation': validation_all['color'].value_counts()}).reindex(LABELS))
print('Validation uses every eligible red/white review:', len(validation_all))

Model 6 training rows retained: 10000
Cross-split normalized descriptions excluded from training: 0


,training,validation
color,,
red,13000,20764
white,7000,10850


Validation uses every eligible red/white review: 31614


## 3. Compare three TF-IDF classifiers

Keep Model 5's unigram/bigram TF-IDF settings and classifier settings. Fit the vectorizer separately on each model's training reviews; only the training size changes. Report training and validation accuracy, macro F1, per-class precision/recall, and confusion matrices.

In [3]:
def text_features():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=10_000, sublinear_tf=True)

models = {
    'Naive Bayes': make_pipeline(text_features(), MultinomialNB(alpha=1.0)),
    'Logistic regression': make_pipeline(text_features(), LogisticRegression(C=1.0, max_iter=2000, random_state=SEED)),
    'Linear SVM': make_pipeline(text_features(), LinearSVC(C=1.0, max_iter=5000, random_state=SEED)),
}
X_train, y_train = train_sample['description'], train_sample['color']
X_val, y_val = validation_all['description'], validation_all['color']
score_rows = []
validation_predictions = {}

def record_scores(name, train_true, train_pred, val_true, val_pred):
    for split, actual, predicted in [('training', train_true, train_pred), ('validation', val_true, val_pred)]:
        score_rows.append({'model': name, 'split': split, 'n': len(actual), 'accuracy': accuracy_score(actual, predicted), 'macro_f1': f1_score(actual, predicted, labels=LABELS, average='macro', zero_division=0)})
    validation_predictions[name] = np.asarray(val_pred)

for name, model in models.items():
    model.fit(X_train, y_train)
    record_scores(name, y_train, model.predict(X_train), y_val, model.predict(X_val))
    print('Finished', name)

Finished Naive Bayes
Finished Logistic regression
Finished Linear SVM


In [4]:
def show_comparison():
    scores = pd.DataFrame(score_rows).pivot(index='model', columns='split', values=['accuracy', 'macro_f1'])
    scores.columns = [f'{metric}_{split}' for metric, split in scores.columns]
    scores['accuracy_gap'] = scores['accuracy_training'] - scores['accuracy_validation']
    scores['macro_f1_gap'] = scores['macro_f1_training'] - scores['macro_f1_validation']
    display(scores.sort_values('macro_f1_validation', ascending=False).round(3))
    for name, predicted in validation_predictions.items():
        print(f'\n{name}: validation report')
        print(classification_report(y_val, predicted, labels=LABELS, zero_division=0, digits=3))
        display(pd.DataFrame(confusion_matrix(y_val, predicted, labels=LABELS), index=['actual red', 'actual white'], columns=['predicted red', 'predicted white']))
    return scores

comparison = show_comparison()

,accuracy_training,accuracy_validation,macro_f1_training,macro_f1_validation,accuracy_gap,macro_f1_gap
model,,,,,,
Linear SVM,0.999,0.972,0.999,0.968,0.028,0.031
Logistic regression,0.994,0.965,0.994,0.960,0.030,0.033
Naive Bayes,0.989,0.964,0.988,0.959,0.026,0.029



Naive Bayes: validation report
              precision    recall  f1-score   support

         red      0.958     0.988     0.973     20764
       white      0.975     0.918     0.945     10850

    accuracy                          0.964     31614
   macro avg      0.967     0.953     0.959     31614
weighted avg      0.964     0.964     0.963     31614



,predicted red,predicted white
actual red,20507,257
actual white,892,9958



Logistic regression: validation report
              precision    recall  f1-score   support

         red      0.957     0.991     0.974     20764
       white      0.981     0.915     0.947     10850

    accuracy                          0.965     31614
   macro avg      0.969     0.953     0.960     31614
weighted avg      0.965     0.965     0.965     31614



,predicted red,predicted white
actual red,20573,191
actual white,917,9933



Linear SVM: validation report
              precision    recall  f1-score   support

         red      0.970     0.988     0.979     20764
       white      0.976     0.941     0.958     10850

    accuracy                          0.972     31614
   macro avg      0.973     0.965     0.968     31614
weighted avg      0.972     0.972     0.972     31614



,predicted red,predicted white
actual red,20508,256
actual white,635,10215


### Observed local TF-IDF results

All three models trained on **20,000 California reviews** (13,000 red; 7,000 white) and used the same **31,614-review all-region red/white validation set** (20,764 red; 10,850 white) as Model 7.2. The 10,000 Model 6 training reviews are retained. DistilBERT has not been run here.

| Model | Training accuracy | Validation accuracy | Validation macro F1 | Macro F1 gap |
|---|---:|---:|---:|---:|
| Naive Bayes | 98.93% | 96.37% | 0.959 | 0.029 |
| Logistic regression | 99.45% | 96.50% | 0.961 | 0.034 |
| Linear SVM | 99.94% | 97.18% | 0.969 | 0.031 |

The gap includes a **geographic distribution shift** between California-only training and all-region validation. Compare with Model 7.2 on this identical validation set before attributing it to model variance.

### Bias–variance and generalization decisions

- **Naive Bayes:** Compare training and validation macro F1; if both are weak, check features and smoothing (`alpha`).
- **Logistic regression:** A large gap suggests stronger regularization (smaller `C`) or more representative data; low scores on both suggest feature limitations.
- **Linear SVM:** A large gap suggests a smaller `C` or more representative data. Evaluate any change by validation macro F1, not the gap alone.
- **DistilBERT:** Compare against the simpler models after the Colab run. More data may help, but inspect validation macro F1 by epoch before changing epochs or weight decay.

Training and validation have different regional compositions in Model 7.1. A gap can therefore reflect **geographic shift**, not only overfitting. The same full red/white validation set is used in both Model 7.1 and Model 7.2; report class-level and geography-level errors alongside aggregate accuracy.

## 4. Train DistilBERT on Google Colab and download results

This section runs only inside Colab and requires a GPU. It uses the same 20,000 training descriptions and the full red/white validation descriptions as the TF-IDF models above. The pretrained DistilBERT weights are fine-tuned for three epochs, with the best checkpoint selected by validation macro F1. After training, the notebook exports row-level training and validation predictions, training history, and data fingerprints in a small ZIP. The model weights remain in Colab; the ZIP has everything needed to compare scores here.

The next cell installs the BERT dependencies only in Colab. After its first install or upgrade, **restart the Colab runtime and rerun from the top** before importing Transformers. Package upgrades in a live kernel can cause stale-import errors.

In [5]:
from importlib.util import find_spec
from importlib import metadata
IN_COLAB = find_spec('google.colab') is not None
if IN_COLAB:
    # Install only in Colab; stop if package versions changed so imports use a fresh runtime.
    packages = ('torch', 'transformers', 'accelerate', 'datasets', 'pyarrow')
    def installed_version(package):
        try:
            return metadata.version(package)
        except metadata.PackageNotFoundError:
            return None
    before = {package: installed_version(package) for package in packages}
    get_ipython().run_line_magic('pip', '-q install -U "transformers==4.57.6" "accelerate>=1,<2" "datasets>=4,<5"')
    after = {package: installed_version(package) for package in packages}
    for package, version in after.items():
        print(f'{package}: {version or "missing"}')
    if before != after:
        raise RuntimeError('Colab packages changed. Restart the runtime and rerun this notebook from the top.')
else:
    print('Local run: Colab training will be skipped. Use Section 5 to import its results ZIP.')

ModuleNotFoundError: No module named 'google'

In [ ]:
if IN_COLAB:
    import torch
    from torch.utils.data import Dataset
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments, set_seed


    set_seed(SEED)
    print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else ('Apple MPS' if torch.backends.mps.is_available() else 'CPU'))
    MODEL_NAME = 'distilbert/distilbert-base-uncased'
    label2id = {'red': 0, 'white': 1}
    id2label = {v: k for k, v in label2id.items()}
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    class ReviewDataset(Dataset):
        def __init__(self, frame):
            self.encodings = tokenizer(frame['description'].tolist(), truncation=True, max_length=128)
            self.labels = frame['color'].map(label2id).astype(int).tolist()
        def __len__(self):
            return len(self.labels)
        def __getitem__(self, index):
            return {**{key: torch.tensor(values[index]) for key, values in self.encodings.items()}, 'labels': torch.tensor(self.labels[index])}

    bert_train = ReviewDataset(train_sample)
    bert_val = ReviewDataset(validation_all)
else:
    print("Skipped DistilBERT training: run this section on a Colab GPU.")

In [ ]:
if IN_COLAB:
    def bert_metrics(result):
        predicted = result.predictions.argmax(axis=-1)
        return {'accuracy': accuracy_score(result.label_ids, predicted), 'macro_f1': f1_score(result.label_ids, predicted, labels=[0, 1], average='macro', zero_division=0)}

    bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id)
    use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 and torch.cuda.is_bf16_supported()
    bert_args = TrainingArguments(
        output_dir=f'modeling_{MODEL_NUMBER}_bert_checkpoints',
        learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=32,
        num_train_epochs=3, weight_decay=0.01,
        eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
        metric_for_best_model='macro_f1', greater_is_better=True, save_total_limit=1,
        fp16=torch.cuda.is_available() and not use_bf16, bf16=use_bf16,
        report_to='none', seed=SEED,
    )
    bert_trainer = Trainer(
        model=bert_model, args=bert_args, train_dataset=bert_train, eval_dataset=bert_val,
        processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=bert_metrics,
    )
    bert_trainer.train()
else:
    print("Skipped DistilBERT training: run this section on a Colab GPU.")

In [ ]:
if IN_COLAB:
    import hashlib
    import io
    import json
    import zipfile
    from google.colab import files

    def frame_fingerprint(frame):
        rows = [[int(row_id), str(row['color']), str(row['description'])]
                for row_id, row in frame.sort_index().iterrows()]
        payload = json.dumps(rows, ensure_ascii=False, separators=(',', ':')).encode('utf-8')
        return hashlib.sha256(payload).hexdigest()

    train_output = bert_trainer.predict(bert_train)
    val_output = bert_trainer.predict(bert_val)
    train_pred = [id2label[int(i)] for i in train_output.predictions.argmax(axis=-1)]
    val_pred = [id2label[int(i)] for i in val_output.predictions.argmax(axis=-1)]
    train_table = pd.DataFrame({'row_id': train_sample.index.astype(int),
                                'actual_color': train_sample['color'].to_numpy(),
                                'predicted_color': train_pred})
    val_table = pd.DataFrame({'row_id': validation_all.index.astype(int),
                              'actual_color': validation_all['color'].to_numpy(),
                              'predicted_color': val_pred})
    export_metadata = {
        'experiment': 'modeling_7_1_distilbert',
        'model': MODEL_NAME,
        'seed': SEED,
        'training_rows': len(train_sample),
        'validation_rows': len(validation_all),
        'training_fingerprint': frame_fingerprint(train_sample),
        'validation_fingerprint': frame_fingerprint(validation_all),
        'best_checkpoint': bert_trainer.state.best_model_checkpoint,
        'training_accuracy': accuracy_score(y_train, train_pred),
        'training_macro_f1': f1_score(y_train, train_pred, labels=LABELS, average='macro', zero_division=0),
        'validation_accuracy': accuracy_score(y_val, val_pred),
        'validation_macro_f1': f1_score(y_val, val_pred, labels=LABELS, average='macro', zero_division=0),
        'training_history': bert_trainer.state.log_history,
    }
    archive_path = Path('modeling_7_1_distilbert_results.zip')
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        archive.writestr('metadata.json', json.dumps(export_metadata, indent=2))
        archive.writestr('training_predictions.csv', train_table.to_csv(index=False))
        archive.writestr('validation_predictions.csv', val_table.to_csv(index=False))
    print(f'Saved {archive_path} ({archive_path.stat().st_size / 2**20:.2f} MiB)')
    print('Validation macro F1:', round(export_metadata['validation_macro_f1'], 4))
    files.download(str(archive_path))
else:
    print('Local run: no BERT training; import the downloaded results in Section 5.')

## 5. Load Colab results into the local comparison

Put `modeling_7_1_distilbert_results.zip` in the same folder as this notebook. This cell reads the ZIP directly without extracting files. It checks the exact selected review IDs, labels, and SHA-256 fingerprints of review text against the local data. If they match, it adds DistilBERT training/validation scores and its validation confusion matrix to the same comparison used above. Rerunning the cell replaces the prior DistilBERT row rather than duplicating it. Until the ZIP is present, the three TF-IDF results remain visible.

In [6]:
import hashlib
import io
import json
import zipfile


def frame_fingerprint(frame):
    rows = [[int(row_id), str(row['color']), str(row['description'])]
            for row_id, row in frame.sort_index().iterrows()]
    payload = json.dumps(rows, ensure_ascii=False, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def verified_predictions(table, frame, name):
    required = {'row_id', 'actual_color', 'predicted_color'}
    if set(table.columns) != required or table.empty or table.isna().any().any():
        raise ValueError(f'{name}: missing, extra, or empty prediction fields')
    if not table['row_id'].is_unique or set(table['row_id']) != set(frame.index):
        raise ValueError(f'{name}: row IDs do not match this notebook')
    aligned = table.set_index('row_id').reindex(frame.index)
    if not aligned['actual_color'].eq(frame['color']).all():
        raise ValueError(f'{name}: actual labels do not match this notebook')
    if not set(aligned['predicted_color']).issubset(LABELS):
        raise ValueError(f'{name}: unexpected predicted label')
    return aligned['predicted_color'].to_numpy()

archive_path = Path('modeling_7_1_distilbert_results.zip')
if archive_path.is_file():
    with zipfile.ZipFile(archive_path) as archive:
        required_files = {'metadata.json', 'training_predictions.csv', 'validation_predictions.csv'}
        if set(archive.namelist()) != required_files:
            raise ValueError('The ZIP does not contain the three expected Model 7.1 result files')
        exported = json.loads(archive.read('metadata.json'))
        saved_train_predictions = pd.read_csv(io.BytesIO(archive.read('training_predictions.csv')))
        saved_val_predictions = pd.read_csv(io.BytesIO(archive.read('validation_predictions.csv')))
    if exported.get('experiment') != 'modeling_7_1_distilbert' or exported.get('model') != 'distilbert/distilbert-base-uncased':
        raise ValueError('The ZIP is not from Model 7.1 DistilBERT')
    if exported.get('seed') != SEED or exported.get('training_rows') != len(train_sample) or exported.get('validation_rows') != len(validation_all):
        raise ValueError('The Colab run used different experiment settings or row counts')
    if exported.get('training_fingerprint') != frame_fingerprint(train_sample):
        raise ValueError('The Colab training reviews differ from this local sample')
    if exported.get('validation_fingerprint') != frame_fingerprint(validation_all):
        raise ValueError('The Colab validation reviews differ from this local validation set')
    bert_train_pred = verified_predictions(saved_train_predictions, train_sample, 'training')
    bert_val_pred = verified_predictions(saved_val_predictions, validation_all, 'validation')
    # Recompute from row-level predictions; do not trust saved summary numbers alone.
    score_rows = [row for row in score_rows if row['model'] != 'DistilBERT']
    validation_predictions.pop('DistilBERT', None)
    record_scores('DistilBERT', y_train, bert_train_pred, y_val, bert_val_pred)
    comparison = show_comparison()
    print('Colab checkpoint:', exported['best_checkpoint'])
    print('Epoch evaluations:')
    display(pd.DataFrame([entry for entry in exported['training_history'] if 'eval_macro_f1' in entry])
            [['epoch', 'eval_accuracy', 'eval_macro_f1']].round(4))
else:
    print(f'No {archive_path.name} yet. Download it from Colab, place it beside this notebook, and rerun this cell.')

,accuracy_training,accuracy_validation,macro_f1_training,macro_f1_validation,accuracy_gap,macro_f1_gap
model,,,,,,
DistilBERT,0.998,0.977,0.998,0.975,0.021,0.023
Linear SVM,0.999,0.972,0.999,0.968,0.028,0.031
Logistic regression,0.994,0.965,0.994,0.960,0.030,0.033
Naive Bayes,0.989,0.964,0.988,0.959,0.026,0.029



Naive Bayes: validation report
              precision    recall  f1-score   support

         red      0.958     0.988     0.973     20764
       white      0.975     0.918     0.945     10850

    accuracy                          0.964     31614
   macro avg      0.967     0.953     0.959     31614
weighted avg      0.964     0.964     0.963     31614



,predicted red,predicted white
actual red,20507,257
actual white,892,9958



Logistic regression: validation report
              precision    recall  f1-score   support

         red      0.957     0.991     0.974     20764
       white      0.981     0.915     0.947     10850

    accuracy                          0.965     31614
   macro avg      0.969     0.953     0.960     31614
weighted avg      0.965     0.965     0.965     31614



,predicted red,predicted white
actual red,20573,191
actual white,917,9933



Linear SVM: validation report
              precision    recall  f1-score   support

         red      0.970     0.988     0.979     20764
       white      0.976     0.941     0.958     10850

    accuracy                          0.972     31614
   macro avg      0.973     0.965     0.968     31614
weighted avg      0.972     0.972     0.972     31614



,predicted red,predicted white
actual red,20508,256
actual white,635,10215



DistilBERT: validation report
              precision    recall  f1-score   support

         red      0.980     0.986     0.983     20764
       white      0.973     0.961     0.967     10850

    accuracy                          0.977     31614
   macro avg      0.976     0.973     0.975     31614
weighted avg      0.977     0.977     0.977     31614



,predicted red,predicted white
actual red,20476,288
actual white,427,10423


Colab checkpoint: modeling_7_1_bert_checkpoints/checkpoint-3750
Epoch evaluations:


,epoch,eval_accuracy,eval_macro_f1
0,1.0,0.9765,0.9740
1,2.0,0.9767,0.9742
2,3.0,0.9774,0.9748


## Result and next decision

Not insignificant differences between train and test, slightly overfit to california wines. we can talka bout this language difference in our presentation/report maybe? minor overfit (for teh same reason), so expansion to train on a variety of regions should help. 